# 04 — Data Quality Checks

Automated checks across Bronze, Silver, and Gold layers.

Each check prints PASS or FAIL. A FAIL does not raise an exception — the full report runs before any action is taken.  
The pipeline's Outlook activity emails a copy of this output on completion.

In [ ]:
results: list[tuple[str, str, str]] = []   # (layer, check, status)


def check(layer: str, description: str, passed: bool) -> None:
    status = "PASS" if passed else "FAIL"
    results.append((layer, description, status))


def scalar(sql: str) -> int:
    return spark.sql(sql).collect()[0][0]


## Bronze Layer

In [ ]:
# Row count expectations (must match data generator config)
check("Bronze", "bronze_hubspot_marketing_sources has 10 rows",
      scalar("SELECT COUNT(*) FROM bronze_hubspot_marketing_sources") == 10)

check("Bronze", "bronze_hubspot_contacts has 1000 rows",
      scalar("SELECT COUNT(*) FROM bronze_hubspot_contacts") == 1000)

check("Bronze", "bronze_hubspot_activities has 5000 rows",
      scalar("SELECT COUNT(*) FROM bronze_hubspot_activities") == 5000)

check("Bronze", "bronze_dynamics365_students has 300 rows",
      scalar("SELECT COUNT(*) FROM bronze_dynamics365_students") == 300)

check("Bronze", "bronze_dynamics365_enrolments has 1000 rows",
      scalar("SELECT COUNT(*) FROM bronze_dynamics365_enrolments") == 1000)

check("Bronze", "bronze_paradigm_results has 1500 rows",
      scalar("SELECT COUNT(*) FROM bronze_paradigm_results") == 1500)

# FK integrity: all D365 enrolments reference real students
check("Bronze", "D365 enrolment.student_number FKs all resolve",
      scalar("""
          SELECT COUNT(*) FROM bronze_dynamics365_enrolments e
          LEFT JOIN bronze_dynamics365_students s ON e.student_number = s.student_number
          WHERE s.student_number IS NULL
      """) == 0)

# FK integrity: D365 students with contact_id all map to real HubSpot contacts
check("Bronze", "D365 student.contact_id FKs all resolve (where not NULL)",
      scalar("""
          SELECT COUNT(*) FROM bronze_dynamics365_students d
          LEFT JOIN bronze_hubspot_contacts h ON d.contact_id = h.contact_id
          WHERE d.contact_id IS NOT NULL AND h.contact_id IS NULL
      """) == 0)

# FK integrity: all Paradigm results reference real Paradigm students and units
check("Bronze", "Paradigm result.student_code FKs all resolve",
      scalar("""
          SELECT COUNT(*) FROM bronze_paradigm_results r
          LEFT JOIN bronze_paradigm_students s ON r.student_code = s.student_code
          WHERE s.student_code IS NULL
      """) == 0)

check("Bronze", "Paradigm result.unit_id FKs all resolve",
      scalar("""
          SELECT COUNT(*) FROM bronze_paradigm_results r
          LEFT JOIN bronze_paradigm_units u ON r.unit_id = u.unit_id
          WHERE u.unit_id IS NULL
      """) == 0)

# No duplicate PKs in bronze
check("Bronze", "bronze_hubspot_contacts: no duplicate contact_id",
      scalar("SELECT COUNT(*) FROM (SELECT contact_id FROM bronze_hubspot_contacts GROUP BY contact_id HAVING COUNT(*) > 1)") == 0)

check("Bronze", "bronze_dynamics365_students: no duplicate student_number",
      scalar("SELECT COUNT(*) FROM (SELECT student_number FROM bronze_dynamics365_students GROUP BY student_number HAVING COUNT(*) > 1)") == 0)


## Silver Layer

In [ ]:
# Row counts match D365 student spine
check("Silver", "silver_student row count matches D365 students (300)",
      scalar("SELECT COUNT(*) FROM silver_student") == 300)

# Identity resolution coverage
hs_link_pct = scalar("""
    SELECT ROUND(COUNT(contact_id) * 100.0 / COUNT(*), 1)
    FROM silver_student
""") 
check("Silver", f"HubSpot link rate is ~90% (actual: {hs_link_pct}%)",
      85 <= hs_link_pct <= 95)

paradigm_link_pct = scalar("""
    SELECT ROUND(COUNT(student_code) * 100.0 / COUNT(*), 1)
    FROM silver_student
""")
check("Silver", f"Paradigm link rate is ~83% (actual: {paradigm_link_pct}%)",
      78 <= paradigm_link_pct <= 88)

# Email is always lowercase in silver_student
check("Silver", "silver_student.email is always lowercase",
      scalar("SELECT COUNT(*) FROM silver_student WHERE email != LOWER(email)") == 0)

# final_grade is NULL (not empty string) for non-completed enrolments
check("Silver", "silver_enrolment.final_grade is NULL for non-completed rows",
      scalar("""
          SELECT COUNT(*) FROM silver_enrolment
          WHERE status != 'completed' AND final_grade IS NOT NULL
      """) == 0)

# silver_result count should be 1500 (all results resolved to a D365 student)
check("Silver", "silver_result has 1500 rows (all Paradigm results resolved)",
      scalar("SELECT COUNT(*) FROM silver_result") == 1500)

# Uniqueness: (student_code, unit_id) in silver_result
check("Silver", "silver_result: no duplicate (student_code, unit_id)",
      scalar("""
          SELECT COUNT(*) FROM (
              SELECT student_code, unit_id
              FROM silver_result
              GROUP BY student_code, unit_id
              HAVING COUNT(*) > 1
          )
      """) == 0)


## Gold Layer

In [ ]:
# Dimension row counts
check("Gold", "dim_student has 300 rows",
      scalar("SELECT COUNT(*) FROM dim_student") == 300)

check("Gold", "dim_course has 20 rows",
      scalar("SELECT COUNT(*) FROM dim_course") == 20)

check("Gold", "dim_intake has 6 rows",
      scalar("SELECT COUNT(*) FROM dim_intake") == 6)

check("Gold", "dim_marketing_source has 10 rows",
      scalar("SELECT COUNT(*) FROM dim_marketing_source") == 10)

# Surrogate key uniqueness
for dim, key in [("dim_student", "student_key"), ("dim_course", "course_key"),
                  ("dim_intake", "intake_key"), ("dim_marketing_source", "marketing_source_key")]:
    dupes = scalar(f"SELECT COUNT(*) FROM (SELECT {key} FROM {dim} GROUP BY {key} HAVING COUNT(*) > 1)")
    check("Gold", f"{dim}.{key} is unique", dupes == 0)

# fact_lead: one row per HubSpot contact
check("Gold", "fact_lead has 1000 rows (one per HubSpot contact)",
      scalar("SELECT COUNT(*) FROM fact_lead") == 1000)

# fact_enrolment: non-NULL required FKs
check("Gold", "fact_enrolment: no NULL student_key, course_key, or intake_key",
      scalar("""
          SELECT COUNT(*) FROM fact_enrolment
          WHERE student_key IS NULL OR course_key IS NULL OR intake_key IS NULL
      """) == 0)

# fact_result: non-NULL required FKs
check("Gold", "fact_result: no NULL student_key, course_key, or intake_key",
      scalar("""
          SELECT COUNT(*) FROM fact_result
          WHERE student_key IS NULL OR course_key IS NULL OR intake_key IS NULL
      """) == 0)

# fact_result grade values are valid
check("Gold", "fact_result.grade only contains valid values (HD/D/C/P/F)",
      scalar("""
          SELECT COUNT(*) FROM fact_result
          WHERE grade NOT IN ('HD', 'D', 'C', 'P', 'F')
      """) == 0)

# marks are in range
check("Gold", "fact_result.mark is in [0, 100]",
      scalar("SELECT COUNT(*) FROM fact_result WHERE mark < 0 OR mark > 100") == 0)


## Summary

In [ ]:
passes = sum(1 for _, _, s in results if s == "PASS")
fails  = sum(1 for _, _, s in results if s == "FAIL")

print(f"{'Layer':<10} {'Check':<65} {'Status'}")
print("-" * 85)
for layer, description, status in results:
    flag = "" if status == "PASS" else "  <-- INVESTIGATE"
    print(f"{layer:<10} {description:<65} {status}{flag}")

print()
print(f"Total: {passes} passed, {fails} failed")

if fails > 0:
    raise RuntimeError(f"{fails} data quality check(s) failed — see report above")
